Importamos librerías

In [ ]:
import os, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import h5py
import torch.optim as optim
from sklearn.manifold import TSNE
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from mpl_toolkits.mplot3d import Axes3D

from matplotlib.animation import FuncAnimation

Creamos directorios para guardar imgs y modelos

In [ ]:
os.makedirs('images', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('./images/t-sne_thumbnails', exist_ok=True)
os.makedirs('gifs', exist_ok=True)

Comprobamos si tenemos GPU / MPS

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
#device = 'mps' if torch.backends.mps.is_available() else 'cpu'
#print(device)

Procesmaos el dataset de MNIST 3D

In [ ]:
class MNISTPointCloudToVoxel(Dataset):
    def __init__(self, h5_path, grid_size=32):
        self.grid_size = grid_size
        self.data = []
        self.labels = []
        
        # Leemos el archivo HDF5
        with h5py.File(h5_path, 'r') as f:
            # Iteramos sobre todos los grupos (las subcarpetas '0', '1', '2'...)
            keys = list(f.keys())
            
            # Como los datasets son pequeños (5000 y 1000 muestras), 
            # es mucho más rápido para entrenar cargarlos todos en la RAM de golpe.
            for k in keys:
                # Extraemos las coordenadas 3D (x, y, z)
                points = f[k]['points'][:]
                # Extraemos la etiqueta original
                label = f[k].attrs['label']
                
                self.data.append(points)
                self.labels.append(label)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        points = self.data[idx]
        label = self.labels[idx]
        
        # Creamos un "cubo" 3D vacío (1 canal, Alto, Ancho, Profundidad)
        voxel = np.zeros((1, self.grid_size, self.grid_size, self.grid_size), dtype=np.float32)
        
        # Mapeamos los puntos (x, y, z) a índices de nuestro cubo
        if len(points) > 0:
            # Encontramos los límites de la nube de puntos para escalarla a nuestro cubo
            p_min = np.min(points, axis=0)
            p_max = np.max(points, axis=0)
            rango = p_max - p_min
            rango[rango == 0] = 1.0 # Evitamos divisiones por cero
            
            # Escalamos todos los puntos para que estén entre 0 y 31 (para grid_size=32)
            points_norm = (points - p_min) / rango
            coords = np.clip(np.round(points_norm * (self.grid_size - 1)), 0, self.grid_size - 1).astype(int)
            
            voxel[0, coords[:, 0], coords[:, 1], coords[:, 2]] = 1.0
            
        # Normalizamos a rango [-1, 1[]
        voxel = (voxel - 0.5) / 0.5
            
        return torch.tensor(voxel), torch.tensor(label, dtype=torch.long)


train_dataset = MNISTPointCloudToVoxel('./data/train_small.h5', grid_size=32)
valid_dataset = MNISTPointCloudToVoxel('./data/valid_small.h5', grid_size=32)
test_dataset = MNISTPointCloudToVoxel('./data/test_small.h5', grid_size=32)

trainloader = DataLoader(train_dataset, batch_size=128, shuffle=True)
valloader = DataLoader(valid_dataset, batch_size=128, shuffle=False)
testloader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# Definimos el VAE

In [ ]:
class MnistEncoder3D(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            # Entrada: [Batch, 1, 32, 32, 32] -> Salida: [Batch, 32, 16, 16, 16]
            nn.Conv3d(in_channels=1, out_channels=32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm3d(num_features=32),
            nn.LeakyReLU(negative_slope=0.2, inplace=True),

            # Entrada: [Batch, 32, 16, 16, 16] -> Salida: [Batch, 64, 8, 8, 8]
            nn.Conv3d(in_channels=32, out_channels=64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm3d(num_features=64),
            nn.LeakyReLU(0.2, True),

            # Entrada: [Batch, 64, 8, 8, 8] -> Salida: [Batch, 128, 4, 4, 4]
            nn.Conv3d(in_channels=64, out_channels=128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm3d(num_features=128),
            nn.LeakyReLU(0.2, True)
        )
        
        # Al aplanar el volumen de 128 canales de 4x4x4 obtenemos 128 * 4 * 4 * 4 = 8192 características
        self.flatten_size = 128 * 4 * 4 * 4
        
        # Capas lineales para obtener la media (mu) y el logaritmo de la varianza (logvar)
        self.fc_mu = nn.Linear(in_features=self.flatten_size, out_features=latent_dim)
        self.fc_logvar = nn.Linear(in_features=self.flatten_size, out_features=latent_dim)

    def forward(self, x):
        h = self.net(x)
        h = h.view(x.size(0), -1) # Aplanamos el tensor
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

In [ ]:
class MnistDecoder3D(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.flatten_size = 128 * 4 * 4 * 4
        
        # Capa lineal para proyectar el vector latente de vuelta a la dimensión aplanada
        self.fc = nn.Linear(in_features=latent_dim, out_features=self.flatten_size)
        
        # Red convolucional transpuesta en 3D para reconstruir el volumen
        self.net = nn.Sequential(
            # Entrada: [Batch, 128, 4, 4, 4] -> Salida: [Batch, 64, 8, 8, 8]
            nn.ConvTranspose3d(in_channels=128, out_channels=64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm3d(num_features=64),
            nn.LeakyReLU(negative_slope=0.2, inplace=True),

            # Entrada: [Batch, 64, 8, 8, 8] -> Salida: [Batch, 32, 16, 16, 16]
            nn.ConvTranspose3d(in_channels=64, out_channels=32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm3d(num_features=32),
            nn.LeakyReLU(0.2, True),

            # Entrada: [Batch, 32, 16, 16, 16] -> Salida: [Batch, 1, 32, 32, 32]
            nn.ConvTranspose3d(in_channels=32, out_channels=1, kernel_size=4, stride=2, padding=1)
            # Nota: No usamos activación al final (o usamos Tanh si los datos están normalizados entre -1 y 1)
        )

    def forward(self, z):
        h = self.fc(z)
        # Redimensionamos a un volumen 3D: [Batch, Canales, Profundidad, Altura, Anchura]
        h = h.view(z.size(0), 128, 4, 4, 4)
        x_logits = self.net(h)
        
        # Usamos tanh porque en nuestro Dataset normalizamos los datos a [-1, 1]
        return torch.tanh(x_logits)


In [ ]:
class MnistVAE3D(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.latent_dim = latent_dim
        self.enc = MnistEncoder3D(latent_dim)
        self.dec = MnistDecoder3D(latent_dim)

    def reparametrize(self, mu, logvar):
        """
        El 'Reparameterization Trick' permite que el gradiente fluya a través del muestreo aleatorio.
        """
        # Desviación estándar es exp(0.5 * logvar)
        std = torch.exp(0.5 * logvar)
        # Muestreamos epsilon de una distribución normal estándar
        eps = torch.randn_like(std)
        # Retornamos la muestra escalada y desplazada
        return mu + eps * std

    def forward(self, x):
        # Codificamos la imagen 3D en los parámetros de la distribución latente
        mu, logvar = self.enc(x)
        
        # Muestreamos del espacio latente
        z = self.reparametrize(mu, logvar)
        
        # Decodificamos el vector de vuelta a una imagen 3D
        recon = self.dec(z)
        
        # Devolvemos la reconstrucción y los parámetros para poder calcular la pérdida (Loss)
        return recon, mu, logvar

## Early Stopping

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0, path='checkpoint.pth'):
        self.patience = patience
        self.min_delta = min_delta
        self.path = path
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(model)

        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True

        else:
            self.best_loss = val_loss
            self.save_checkpoint(model)
            self.counter = 0

    def save_checkpoint(self, model):
        torch.save(model.state_dict(), self.path)

## Función pérdida del VAE

In [ ]:
def vae_loss(recon_x, x, mu, logvar, beta=1.0):
    # Tamaño del batch
    batch_size = x.size(0)
    
    # MSE Loss: calculamos el error sin reducir, sumamos por cada volumen 3D, y hacemos la media del batch
    recon_tensor = F.mse_loss(recon_x, x, reduction='none')
    recon = recon_tensor.view(batch_size, -1).sum(dim=1).mean()
    
    # KL Divergence: sumamos por dimensiones latentes, y hacemos la media del batch
    kl = -0.5 * torch.sum(1 + logvar - mu ** 2 - logvar.exp(), dim=1).mean()
    
    # Pérdida total ponderada
    return recon + beta * kl, recon, kl

# Train Models

In [ ]:
def _plot_training_results(history, title):
    def normalize(data):
        d = np.array(data)
        return (d - d.min()) / (d.max() - d.min() + 1e-8)

    ep_rng = range(1, len(history['loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    axes[0].plot(ep_rng, normalize(history['rec']), label='Recon (Norm)', color='green')
    axes[0].plot(ep_rng, normalize(history['kl']), label='KL (Norm)', color='red')
    axes[0].set_title("Dinámica Normalizada: Recon vs KL vs Beta")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(ep_rng, history['rec'], label='MSE Real', color='green', linewidth=2)
    axes[1].set_title("Calidad de Reconstrucción (Sin normalizar)")
    axes[1].set_ylabel("Suma de Errores Cuadráticos")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(f'./images/{title}.png') 
    plt.show()

In [ ]:
def train_vae(model, optimizer, train_loader, val_loader,
                            epochs=100, patience=20, title="VAE_3D"):

    history = {'loss': [], 'rec': [], 'kl': [], 'beta': []}
    early_stopping = EarlyStopping(patience=patience, path=f'./models/{title}_best.pth') 

    for ep in range(1, epochs + 1):
        model.train()
        train_loss, train_rec, train_kl = 0.0, 0.0, 0.0

        for x, _ in train_loader:
            x = x.to(device)
            recon, mu, logvar = model(x)

            loss, rec, kl = vae_loss(recon, x, mu, logvar)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_rec += rec.item()
            train_kl += kl.item()

        n = len(train_loader)
        history['loss'].append(train_loss / n)
        history['rec'].append(train_rec / n)
        history['kl'].append(train_kl / n)
        
        print(f"Epoch {ep:02d}/{epochs} | Rec: {history['rec'][-1]:.2f} | KL: {history['kl'][-1]:.2f}")

        model.eval()
        with torch.no_grad():
            x_test, _ = next(iter(val_loader))
            x_test = x_test[:16].to(device)
            recon_test, _, _ = model(x_test)

            def denorm(t): return (t * 0.5 + 0.5).clamp(0,1)

            x_img = denorm(x_test).cpu().numpy()
            recon_img = denorm(recon_test).cpu().numpy()
            
            slice_idx = x_img.shape[2] // 2 

            fig, ax = plt.subplots(2, 16, figsize=(32, 4))
            for i in range(16):
                ax[0, i].imshow(x_img[i, 0, slice_idx, :, :], cmap="gray"); ax[0, i].axis("off")
                ax[1, i].imshow(recon_img[i, 0, slice_idx, :, :], cmap="gray"); ax[1, i].axis("off")
            fig.suptitle(f"Epoch {ep} - Reconstrucción paso a paso (Corte 2D)")
            plt.tight_layout()
            plt.show()

        # Validación y Early Stopping
        val_loss_tot = 0
        val_rec_tot = 0

        model.eval()
        with torch.no_grad():
            for xv, _ in val_loader:
                xv = xv.to(device)
                rv, mv, lv = model(xv)

                v_loss, v_rec, v_kl = vae_loss(rv, xv, mv, lv)

                val_loss_tot += v_loss.item()
                val_rec_tot += v_rec.item() 

        avg_val_loss = val_loss_tot / len(val_loader)
        avg_val_rec = val_rec_tot / len(val_loader)

        early_stopping(avg_val_rec, model)

        if early_stopping.early_stop:
            print("Early stopping activado. Recargando mejor modelo.")
            model.load_state_dict(torch.load(early_stopping.path))
            break

    _plot_training_results(history, title)
    return history

### Mnist VAE con latent_dim = 128

In [ ]:
latent_dim = 128
modelo_3d = MnistVAE3D(latent_dim=latent_dim).to(device)

optimizer = optim.Adam(modelo_3d.parameters(), lr=1e-3, weight_decay=1e-3)

history = train_vae(
    model=modelo_3d, 
    optimizer=optimizer, 
    train_loader=trainloader, 
    val_loader=valloader, 
    epochs=100,          
    patience=20,        # Early stopping: se detendrá si no mejora en 20 épocas
    title="VAE_ld_128_3D" 
)

### Mnist VAE con latent_dim = 64

In [ ]:
latent_dim = 64
modelo_3d = MnistVAE3D(latent_dim=latent_dim).to(device)

optimizer = optim.Adam(modelo_3d.parameters(), lr=1e-3, weight_decay=1e-3)

history = train_vae(
    model=modelo_3d, 
    optimizer=optimizer, 
    train_loader=trainloader, 
    val_loader=valloader, 
    epochs=100,          
    patience=20,        # Early stopping: se detendrá si no mejora en 20 épocas
    title="VAE_ld_64_3D" 
)

### Mnist VAE con latent_dim = 48

In [ ]:
latent_dim = 48
modelo_3d = MnistVAE3D(latent_dim=latent_dim).to(device)

optimizer = optim.Adam(modelo_3d.parameters(), lr=1e-3, weight_decay=1e-3)

history = train_vae(
    model=modelo_3d, 
    optimizer=optimizer, 
    train_loader=trainloader, 
    val_loader=valloader, 
    epochs=100,          
    patience=20,        # Early stopping: se detendrá si no mejora en 20 épocas
    title="VAE_ld_48_3D" 
)

### Mnist VAE con latent_dim = 32

In [ ]:
latent_dim = 32
modelo_3d = MnistVAE3D(latent_dim=latent_dim).to(device)

optimizer = optim.Adam(modelo_3d.parameters(), lr=1e-3, weight_decay=1e-3)

history = train_vae(
    model=modelo_3d, 
    optimizer=optimizer, 
    train_loader=trainloader, 
    val_loader=valloader, 
    epochs=100,          
    patience=20,        # Early stopping: se detendrá si no mejora en 20 épocas
    title="VAE_ld_32_3D" 
)

### VAE con latent_dim = 16

In [ ]:
latent_dim = 16
modelo_3d = MnistVAE3D(latent_dim=latent_dim).to(device)

optimizer = optim.Adam(modelo_3d.parameters(), lr=1e-3, weight_decay=1e-3)

history = train_vae(
    model=modelo_3d, 
    optimizer=optimizer, 
    train_loader=trainloader, 
    val_loader=valloader, 
    epochs=100,          
    patience=20,        # Early stopping: se detendrá si no mejora en 20 épocas
    title="VAE_ld_16_3D" 
)

### VAE con latent_dim = 8

In [ ]:
latent_dim = 8
modelo_3d = MnistVAE3D(latent_dim=latent_dim).to(device)

optimizer = optim.Adam(modelo_3d.parameters(), lr=1e-3, weight_decay=1e-3)

history = train_vae(
    model=modelo_3d, 
    optimizer=optimizer, 
    train_loader=trainloader, 
    val_loader=valloader, 
    epochs=100,          
    patience=20,        # Early stopping: se detendrá si no mejora en 20 épocas
    title="VAE_ld_8_3D" 
)

### VAE con latent_dim = 4

In [ ]:
latent_dim = 4
modelo_3d = MnistVAE3D(latent_dim=latent_dim).to(device)

optimizer = optim.Adam(modelo_3d.parameters(), lr=1e-3, weight_decay=1e-3)

history = train_vae(
    model=modelo_3d, 
    optimizer=optimizer, 
    train_loader=trainloader, 
    val_loader=valloader, 
    epochs=100,          
    patience=20,        # Early stopping: se detendrá si no mejora en 20 épocas
    title="VAE_ld_4_3D" 
)

### VAE con latent_dim = 2

In [ ]:
latent_dim = 2
modelo_3d = MnistVAE3D(latent_dim=latent_dim).to(device)

optimizer = optim.Adam(modelo_3d.parameters(), lr=1e-3, weight_decay=1e-3)

history = train_vae(
    model=modelo_3d, 
    optimizer=optimizer, 
    train_loader=trainloader, 
    val_loader=valloader, 
    epochs=100,          
    patience=20,        # Early stopping: se detendrá si no mejora en 20 épocas
    title="VAE_ld_2_3D" 
)

# Carga pesos modelos

In [ ]:
latent_dim = 128  
modelo_ld_128 = MnistVAE3D(latent_dim).to(device)

modelo_ld_128.load_state_dict(torch.load("./models/VAE_ld_128_3D_best.pth", map_location=device))
modelo_ld_128.eval()

In [ ]:
latent_dim = 64  
modelo_ld_64 = MnistVAE3D(latent_dim).to(device)

modelo_ld_64.load_state_dict(torch.load("./models/VAE_ld_64_3D_best.pth", map_location=device))
modelo_ld_64.eval()

In [ ]:
latent_dim = 48
modelo_ld_48 = MnistVAE3D(latent_dim).to(device)

modelo_ld_48.load_state_dict(torch.load("./models/VAE_ld_48_3D_best.pth", map_location=device))
modelo_ld_48.eval()

In [ ]:
latent_dim = 32
modelo_ld_32 = MnistVAE3D(latent_dim).to(device)

modelo_ld_32.load_state_dict(torch.load("./models/VAE_ld_32_3D_best.pth", map_location=device))
modelo_ld_32.eval()

In [ ]:
latent_dim = 16  
modelo_ld_16 = MnistVAE3D(latent_dim).to(device)

modelo_ld_16.load_state_dict(torch.load("./models/VAE_ld_16_3D_best.pth", map_location=device))
modelo_ld_16.eval()

In [ ]:
latent_dim = 8 
modelo_ld_8 = MnistVAE3D(latent_dim).to(device)

modelo_ld_8.load_state_dict(torch.load("./models/VAE_ld_8_3D_best.pth", map_location=device))
modelo_ld_8.eval()

In [ ]:
latent_dim = 4  
modelo_ld_4 = MnistVAE3D(latent_dim).to(device)

modelo_ld_4.load_state_dict(torch.load("./models/VAE_ld_4_3D_best.pth", map_location=device))
modelo_ld_4.eval()

In [ ]:
latent_dim = 2 
modelo_ld_2 = MnistVAE3D(latent_dim).to(device)

modelo_ld_2.load_state_dict(torch.load("./models/VAE_ld_2_3D_best.pth", map_location=device))
modelo_ld_2.eval()

# Interpolation
- Para ello tomamos 2 imágenes del test set, las codificamos, y luego interpolamos linealmente en el espacio latente entre sus representaciones.
- Finalmente decodificamos cada punto de la interpolación para ver la transición entre ambas imágenes.

In [ ]:
def denorm(t): 
    return (t * 0.5 + 0.5).clamp(0,1)

def interpolation_3d(model, dataloader, device, n_interp=15):
    with torch.no_grad():
        model.eval()
        
        # Sacamos un batch del dataloader y nos quedamos con las 2 primeras imágenes
        x, _ = next(iter(dataloader))
        x = x[:2].to(device)
        
        # Pasamos por el encoder
        mu, logvar = model.enc(x)
        z = mu 

        z1, z2 = z[0], z[1]
        
        # Creamos los pasos de interpolación
        alphas = torch.linspace(0, 1, n_interp).to(device)
        interp_z = torch.stack([z1 * (1 - alpha) + z2 * alpha for alpha in alphas], dim=0)
        
        # Decodificamos los vectores interpolados de vuelta a volúmenes 3D
        interp_x = model.dec(interp_z)
        interp_x = denorm(interp_x).cpu().numpy()
        
        slice_idx = 16
        
        # Visualización
        fig, ax = plt.subplots(1, n_interp, figsize=(2*n_interp, 2))
        for i in range(n_interp):
            img_slice = interp_x[i, 0, slice_idx, :, :]
            ax[i].imshow(img_slice, cmap="gray")
            ax[i].axis("off")
            
        plt.suptitle("Interpolación en el espacio latente 3D")
        plt.tight_layout()
        plt.show()

In [ ]:
interpolation_3d(model=modelo_ld_128, dataloader=testloader, device=device)

In [ ]:
interpolation_3d(model=modelo_ld_64, dataloader=testloader, device=device)

In [ ]:
interpolation_3d(model=modelo_ld_48, dataloader=testloader, device=device)

In [ ]:
interpolation_3d(model=modelo_ld_32, dataloader=testloader, device=device)

In [ ]:
interpolation_3d(model=modelo_ld_16, dataloader=testloader, device=device)

In [ ]:
interpolation_3d(model=modelo_ld_8, dataloader=testloader, device=device)

In [ ]:
interpolation_3d(model=modelo_ld_4, dataloader=testloader, device=device)

In [ ]:
interpolation_3d(model=modelo_ld_2, dataloader=testloader, device=device)

# Espacio latente t-SNE 2D

In [ ]:
def denorm(t): 
    return (t * 0.5 + 0.5).clamp(0, 1)

def latent_space_3d(model, dataloader, device, title_p1="Latent_Space_Reals", title_p2="Latent_Space_Prior"):
    model.eval()
    with torch.no_grad():
        latents = []
        labels_list = []
        # Iteramos sobre el dataloader que le pasemos (testloader)
        for x, y in dataloader:
            x = x.to(device)
            # Pasamos por el encoder y nos quedamos con mu (el vector latente)
            mu, _ = model.enc(x) 
            latents.append(mu.cpu())
            labels_list.append(y)
            
        latents = torch.cat(latents, dim=0).numpy()
        labels_list = torch.cat(labels_list, dim=0).numpy()
        
    tsne = TSNE(n_components=2, random_state=42)
    latents_2d = tsne.fit_transform(latents)
    
    plt.figure(figsize=(8, 8))
    for digit in range(10):
        idx = labels_list == digit
        plt.scatter(latents_2d[idx, 0], latents_2d[idx, 1], label=str(digit), alpha=0.5)
    plt.legend()
    plt.title(title_p1)
    plt.savefig(f'./images/{title_p1}.png') 
    plt.show()

    with torch.no_grad():
        n_samples = 1000
        latent_dim = model.enc.fc_mu.out_features
        
        # Muestreamos del espacio latente (prior N(0, I))
        z = torch.randn(n_samples, latent_dim, device=device)
        
        # Decodificamos a volúmenes 3D
        recon = model.dec(z)
        recon_denorm = denorm(recon) # Normalizamos a [0, 1]
        
        # Binarizamos directamente en la GPU 
        # El resultado sigue teniendo forma [1000, 1, 32, 32, 32]
        recon_bin = (recon_denorm > 0.5).float() 
        
        # Volvemos a codificar las imágenes generadas para ver dónde caen 
        # (Usamos model.enc() para sacar el 'mu' directamente, que es la representación correcta)
        mu_gen, _ = model.enc(recon_bin)
        latents_gen = mu_gen.cpu().numpy()
        
    tsne_gen = TSNE(n_components=2, random_state=42)
    latents_gen_2d = tsne_gen.fit_transform(latents_gen)
    
    plt.figure(figsize=(8, 8))
    plt.scatter(latents_gen_2d[:, 0], latents_gen_2d[:, 1], alpha=0.5, color='purple')
    plt.title(title_p2)
    plt.savefig(f'./images/{title_p2}.png') 
    plt.show()


In [ ]:
latent_space_3d(model=modelo_ld_128, dataloader=testloader, device=device, 
                title_p1='t-SNE of VAE MNIST 3D Latent Space 128',
                title_p2='t-SNE of Prior Samples in Latent Space 128 VAE MNIST 3D')

In [ ]:
latent_space_3d(model=modelo_ld_64, dataloader=testloader, device=device, 
                title_p1='t-SNE of VAE MNIST 3D Latent Space 64',
                title_p2='t-SNE of Prior Samples in Latent Space 64 VAE MNIST 3D')

In [ ]:
latent_space_3d(model=modelo_ld_48, dataloader=testloader, device=device, 
                title_p1='t-SNE of VAE MNIST 3D Latent Space 48',
                title_p2='t-SNE of Prior Samples in Latent Space 48 VAE MNIST 3D')

In [ ]:
latent_space_3d(model=modelo_ld_32, dataloader=testloader, device=device, 
                title_p1='t-SNE of VAE MNIST 3D Latent Space 32',
                title_p2='t-SNE of Prior Samples in Latent Space 32 VAE MNIST 3D')

In [ ]:
latent_space_3d(model=modelo_ld_16, dataloader=testloader, device=device, 
                title_p1='t-SNE of VAE MNIST 3D Latent Space 16',
                title_p2='t-SNE of Prior Samples in Latent Space 16 VAE MNIST 3D')

In [ ]:
latent_space_3d(model=modelo_ld_8, dataloader=testloader, device=device, 
                title_p1='t-SNE of VAE MNIST 3D Latent Space 8',
                title_p2='t-SNE of Prior Samples in Latent Space 8 VAE MNIST 3D')

In [ ]:
latent_space_3d(model=modelo_ld_4, dataloader=testloader, device=device, 
                title_p1='t-SNE of VAE MNIST 3D Latent Space 4',
                title_p2='t-SNE of Prior Samples in Latent Space 4 VAE MNIST 3D')

In [ ]:
latent_space_3d(model=modelo_ld_2, dataloader=testloader, device=device, 
                title_p1='t-SNE of VAE MNIST 3D Latent Space 2',
                title_p2='t-SNE of Prior Samples in Latent Space 2 VAE MNIST 3D')

### t-SNE 3D

In [ ]:
def generar_y_descargar_gif_3d(model, dataloader, device, titulo="Latent_Space_3D"):
    model.eval()
    latents = []
    labels_list = []

    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            # Sacamos el vector latente (mu) directamente usando el encoder
            mu, _ = model.enc(x)
            latents.append(mu.cpu())
            labels_list.append(y)
            
        latents = torch.cat(latents, dim=0).numpy()
        labels_list = torch.cat(labels_list, dim=0).numpy()

    n_total = len(latents)
    n_muestras = min(2000, n_total) 
    
    indices = np.random.choice(n_total, n_muestras, replace=False)
    latents_sample = latents[indices]
    labels_sample = labels_list[indices]

    # t-SNE a 3 dimensiones 
    tsne = TSNE(n_components=3, random_state=42)
    latents_3d = tsne.fit_transform(latents_sample)

    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='3d')
    colors = plt.cm.tab10(np.linspace(0, 1, 10))

    def update(frame):
        ax.clear()
        ax.set_box_aspect([1, 1, 1])
        ax.set_axis_off()

        for digit in range(10):
            idx = labels_sample == digit
            ax.scatter(latents_3d[idx, 0],
                       latents_3d[idx, 1],
                       latents_3d[idx, 2],
                       c=[colors[digit]],
                       label=str(digit),
                       s=15,             
                       alpha=0.6)

        ax.set_title(titulo)
        ax.legend(loc='upper left', bbox_to_anchor=(1, 0.9), title="Dígitos")

        # Rotación de la cámara
        ax.view_init(elev=20, azim=frame)
        return fig,

    frames_lentos = np.arange(0, 360, 2)
    ani = FuncAnimation(fig, update, frames=frames_lentos, interval=50)

    ruta_guardado = f'./gifs/{titulo}.gif'
    ani.save(ruta_guardado, writer='pillow', fps=20)
    plt.close()
    

In [ ]:
generar_y_descargar_gif_3d(model=modelo_ld_128, dataloader=testloader, device=device, titulo = 't_SNE_3D_ld_128')

In [ ]:
generar_y_descargar_gif_3d(model=modelo_ld_64, dataloader=testloader, device=device, titulo = 't_SNE_3D_ld_64')

In [ ]:
generar_y_descargar_gif_3d(model=modelo_ld_48, dataloader=testloader, device=device, titulo = 't_SNE_3D_ld_48')

In [ ]:
generar_y_descargar_gif_3d(model=modelo_ld_32, dataloader=testloader, device=device, titulo = 't_SNE_3D_ld_32')

In [ ]:
generar_y_descargar_gif_3d(model=modelo_ld_16, dataloader=testloader, device=device, titulo = 't_SNE_3D_ld_16')

In [ ]:
generar_y_descargar_gif_3d(model=modelo_ld_8, dataloader=testloader, device=device, titulo = 't_SNE_3D_ld_8')

In [ ]:
generar_y_descargar_gif_3d(model=modelo_ld_4, dataloader=testloader, device=device, titulo = 't_SNE_3D_ld_4')

## t-SNE con reconstrucciones

In [ ]:
@torch.no_grad()
def denorm_3d(t):  # [-1,1] -> [0,1]
    return (t * 0.5 + 0.5).clamp(0, 1)

def pick_spread_points(emb, n_show=150, seed=42):
    rng = np.random.default_rng(seed)
    idx = np.arange(emb.shape[0])
    rng.shuffle(idx)

    chosen = []
    thr = 0.02 * np.var(emb, axis=0).sum()
    for i in idx:
        if not chosen:
            chosen.append(i); continue
        d2 = np.min(np.sum((emb[chosen] - emb[i])**2, axis=1))
        if d2 > thr:
            chosen.append(i)
        if len(chosen) >= n_show:
            break
    return np.array(chosen)

@torch.no_grad()
def tsne_with_recon_thumbnails_3d(model, loader, device, n_total=1000, n_show=150, zoom=0.8, seed=42, title="tSNE_Thumbnails_3D"):
    model.eval()

    X = []
    MU = []
    RECON = []

    for x, _ in loader:
        x = x.to(device)
        recon, mu, logvar = model(x)
        X.append(x.detach().cpu())
        MU.append(mu.detach().cpu())
        RECON.append(recon.detach().cpu())
        if sum(t.size(0) for t in MU) >= n_total:
            break

    X = torch.cat(X, dim=0)[:n_total]
    MU = torch.cat(MU, dim=0)[:n_total].numpy()
    RECON = torch.cat(RECON, dim=0)[:n_total]

    emb = TSNE(n_components=2, random_state=seed, init="pca", learning_rate="auto").fit_transform(MU)
    
    n_show_actual = min(n_show, len(MU))
    chosen = pick_spread_points(emb, n_show=n_show_actual, seed=seed)

    slice_idx = 16
    thumbs = denorm_3d(RECON[chosen])[:, 0, slice_idx, :, :].numpy()  # Queda (n_show, 32, 32)

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_title(f"{title}")
    ax.set_xticks([]); ax.set_yticks([])

    for k, i in enumerate(chosen):
        ab = AnnotationBbox(OffsetImage(thumbs[k], zoom=zoom, cmap="gray"),
                            (emb[i,0], emb[i,1]), frameon=False)
        ax.add_artist(ab)

    ax.set_xlim(emb[chosen,0].min()-5, emb[chosen,0].max()+5)
    ax.set_ylim(emb[chosen,1].min()-5, emb[chosen,1].max()+5)
    plt.tight_layout()
    
    plt.savefig(f'./images/t-sne_thumbnails/{title}.png')
    plt.show()


In [ ]:
tsne_with_recon_thumbnails_3d(model=modelo_ld_128, loader=testloader, device=device, 
                              n_total=3000, n_show=150, zoom=0.9, title='t-SNE of VAE MNIST 3D LS 128 Img')

In [ ]:
tsne_with_recon_thumbnails_3d(model=modelo_ld_64, loader=testloader, device=device, 
                              n_total=3000, n_show=150, zoom=0.9, title='t-SNE of VAE MNIST 3D LS 64 Img')

In [ ]:
tsne_with_recon_thumbnails_3d(model=modelo_ld_48, loader=testloader, device=device, 
                              n_total=3000, n_show=150, zoom=0.9, title='t-SNE of VAE MNIST 3D LS 48 Img')

In [ ]:
tsne_with_recon_thumbnails_3d(model=modelo_ld_32, loader=testloader, device=device, 
                              n_total=3000, n_show=150, zoom=0.9, title='t-SNE of VAE MNIST 3D LS 32 Img')

In [ ]:
tsne_with_recon_thumbnails_3d(model=modelo_ld_16, loader=testloader, device=device, 
                              n_total=3000, n_show=150, zoom=0.9, title='t-SNE of VAE MNIST 3D LS 16 Img')

In [ ]:
tsne_with_recon_thumbnails_3d(model=modelo_ld_8, loader=testloader, device=device, 
                              n_total=3000, n_show=150, zoom=0.9, title='t-SNE of VAE MNIST 3D LS 8 Img')

In [ ]:
tsne_with_recon_thumbnails_3d(model=modelo_ld_4, loader=testloader, device=device, 
                              n_total=3000, n_show=150, zoom=0.9, title='t-SNE of VAE MNIST 3D LS 4 Img')

In [ ]:
tsne_with_recon_thumbnails_3d(model=modelo_ld_2, loader=testloader, device=device, 
                              n_total=3000, n_show=150, zoom=0.9, title='t-SNE of VAE MNIST 3D LS 2 Img')